# SNU AI Challenge - Basic CV Augmented 5-Fold Experiment

This experiment is isolated from EXP-003. It reuses the 9,535 original frozen representations, extracts only the 6,445 basic-CV augmented rows with batched GPU forwards, validates on unmodified original rows only, and writes new checkpoints to a separate Drive result root.

## 1. Configuration and mandatory GPU confirmation

Before setting `GPU_CONFIG_CONFIRMED = True`, confirm the detected GPU, VRAM, feature batch size, image worker count, and training batch size with the user. Start with feature batch 32 on an A100 40 GB; increase only after the GPU telemetry log shows headroom.

In [5]:
import os
from pathlib import Path

DRIVE_ROOT = Path(os.environ.get("SNU_DRIVE_ROOT", "/content/drive/MyDrive/snu ai"))
WORKDIR = Path(os.environ.get("SNU_WORKDIR", "/content/snu_ai_augmented_v1"))
FULL_BUNDLE = DRIVE_ROOT / "snu_colab_full_train_bundle.tar"
AUGMENTED_BUNDLE = DRIVE_ROOT / "snu_colab_basic_cv_augmented_bundle.tar"
CODE_BUNDLE = DRIVE_ROOT / "snu_colab_augmented_training_code.tar"
RAW_CACHE_SHARDS = DRIVE_ROOT / "cache/full_train_9535_shards"
AUGMENTED_CACHE_SHARDS = DRIVE_ROOT / "cache/basic_cv_augmented_6445_batched_v1"
RESULTS_ROOT = DRIVE_ROOT / "results/basic_cv_augmented_resident_v1"
COMBINED_CACHE_ROOT = WORKDIR / "feature_cache/basic_cv_augmented_combined"

GPU_CONFIG_CONFIRMED = True
FEATURE_BATCH_SIZE = 32
IMAGE_WORKERS = 16
TRAINING_BATCH_SIZE = 128
EVALUATION_BATCH_SIZE = 256
FOLDS_TO_RUN = [0, 1, 2, 3, 4]


## 2. Mount Drive, install dependencies, and extract isolated bundles

In [6]:
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "transformers==5.13.0", "accelerate==1.14.0", "pillow>=10"], check=True)
for path in (FULL_BUNDLE, AUGMENTED_BUNDLE, CODE_BUNDLE):
    if not path.is_file():
        raise FileNotFoundError(path)
WORKDIR.mkdir(parents=True, exist_ok=True)
if not (WORKDIR / "snuaichallenge_data/train.csv").is_file():
    subprocess.run(["tar", "-xf", str(FULL_BUNDLE), "-C", str(WORKDIR)], check=True)
if not (WORKDIR / "augmentation/generated/basic_cv/train_augmented.csv").is_file():
    subprocess.run(["tar", "-xf", str(AUGMENTED_BUNDLE), "-C", str(WORKDIR)], check=True)
subprocess.run(["tar", "-xf", str(CODE_BUNDLE), "-C", str(WORKDIR)], check=True)
AUGMENTED_CACHE_SHARDS.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(WORKDIR, AUGMENTED_CACHE_SHARDS, RESULTS_ROOT, sep="\n")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/snu_ai_augmented_v1
/content/drive/MyDrive/snu ai/cache/basic_cv_augmented_6445_batched_v1
/content/drive/MyDrive/snu ai/results/basic_cv_augmented_resident_v1


## 3. Confirm GPU settings with the user

Run this inspection cell, discuss the output, then change `GPU_CONFIG_CONFIRMED` to `True` in the configuration cell. The experiment runner refuses to start otherwise.

In [7]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required")
props = torch.cuda.get_device_properties(0)
print({"gpu": props.name, "vram_gib": props.total_memory / 1024**3, "feature_batch": FEATURE_BATCH_SIZE, "image_workers": IMAGE_WORKERS, "training_batch": TRAINING_BATCH_SIZE, "evaluation_batch": EVALUATION_BATCH_SIZE})
if not GPU_CONFIG_CONFIRMED:
    raise RuntimeError("Confirm these GPU settings with the user before caching or training.")


{'gpu': 'NVIDIA A100-SXM4-40GB', 'vram_gib': 39.4935302734375, 'feature_batch': 32, 'image_workers': 16, 'training_batch': 128, 'evaluation_batch': 256}


## 4. Cache only the augmented 6,445 rows

Original EXP-003 representations are restored and reused. Qwen and ConvNeXt process multiple samples per forward pass. Cache scans report progress every 500 samples, and feature extraction reports completed/total, percentage, remaining samples, throughput, elapsed time, and ETA after every GPU batch. GPU telemetry is saved as `feature_cache_gpu_dmon.log` under the new result root.

In [8]:
common_command = [
    sys.executable, str(WORKDIR / "scripts/run_augmented_5fold_colab.py"),
    "--original-csv", str(WORKDIR / "snuaichallenge_data/splits/full_folds_5.csv"),
    "--original-image-root", str(WORKDIR / "snuaichallenge_data/train"),
    "--augmented-csv", str(WORKDIR / "augmentation/generated/basic_cv/train_augmented.csv"),
    "--augmented-image-root", str(WORKDIR / "augmentation/generated/basic_cv/images"),
    "--combined-cache-root", str(COMBINED_CACHE_ROOT),
    "--raw-cache-shards", str(RAW_CACHE_SHARDS),
    "--augmented-cache-shards", str(AUGMENTED_CACHE_SHARDS),
    "--results-root", str(RESULTS_ROOT),
    "--feature-batch-size", str(FEATURE_BATCH_SIZE),
    "--image-workers", str(IMAGE_WORKERS),
    "--training-batch-size", str(TRAINING_BATCH_SIZE),
    "--evaluation-batch-size", str(EVALUATION_BATCH_SIZE),
    "--folds", *map(str, FOLDS_TO_RUN),
    "--confirm-gpu-config",
]
subprocess.run([*common_command, "--cache-only"], cwd=WORKDIR, check=True)


KeyboardInterrupt: 

## 5. Run leakage-safe five-fold training and one final scorer

Each fold validates only on its unmodified original rows. An augmented row is included in training only when its source original is outside the validation fold. The five fold checkpoints are validation artifacts and must not be ensembled.

In [ ]:
subprocess.run(common_command, cwd=WORKDIR, check=True)


## Expected isolated outputs

- `MyDrive/snu ai/cache/basic_cv_augmented_6445_batched_v1/`: augmented-only cache shards and manifest
- `MyDrive/snu ai/results/basic_cv_augmented_resident_v1/fold_*/`: five validation artifacts
- `MyDrive/snu ai/results/basic_cv_augmented_resident_v1/five_fold_summary.json`: comparable original-only validation result
- `MyDrive/snu ai/results/basic_cv_augmented_resident_v1/final_scorer/final.pt`: one final model trained on 9,535 originals plus 6,445 augmented rows

After completion, record the measured result and likely cause in `MODELS.md`.